In [ ]:
import os
import scanpy as sc

In [ ]:
patient_id = "P2CRC"
data_type = "Xenium"
sc_ref_file = "adata_sc_all_reanno.h5ad"

cell_type_col = "Level1"

output_dir = f"output/sc_SVC_case_visium/{patient_id}_{data_type}"
os.makedirs(output_dir, exist_ok=True)

raw_data_path = "./raw_data/Sim2Real-ST/spot_size"
st_file_name = f"{raw_data_path}/{patient_id}/cut_part1/spot_100/xenium_spot.h5ad"
sc_file_name = f"{raw_data_path}/{patient_id}/cut_part1/real_sc_ref_all.h5ad"

In [ ]:
st_adata = sc.read(st_file_name)
sc_ref_adata = sc.read(sc_file_name)
sc_ref_adata = sc_ref_adata[sc_ref_adata.obs['Patient'] == patient_id, :]
sc_ref_adata.obs['Level1'].replace({"Mono/Macro": "Mono_Macro"}, inplace=True)
sc_ref_adata.obs['celltype1'].replace({"Mono/Macro": "Mono_Macro"}, inplace=True)
sc_ref_adata.obs['clusters'].replace({"Mono/Macro": "Mono_Macro"}, inplace=True)

In [ ]:
from revise.conf.application_sc_sr_conf import ApplicationScSrConf
from revise.application import ScSVCSr
from revise.tools.log import Logger

config = ApplicationScSrConf(
    sample_name=patient_id,
    raw_data_path=raw_data_path,
    result_root_path=output_dir,
    cell_type_col=cell_type_col,
    confidence_col="Confidence",
    unknown_key="Unknown",
    st_file=f"{data_type}.h5ad",
    sc_ref_file=sc_ref_file
)

logger = Logger(name=f"run.log", log_file=f'{output_dir}/application_sc.log').get_logger()
sc_svc = ScSVCSr(st_adata, sc_ref_adata, config, logger)

In [ ]:
sc_svc.global_anchoring()
sc_svc.local_refinement()

In [ ]:
sc_svc_dec = sc_svc.svc["sc_svc_dec"].copy()
sc.pp.neighbors(sc_svc_dec, n_neighbors=30)
sc.tl.umap(sc_svc_dec)

sc.pl.umap(
    sc_svc_dec,
    color="cell_type",
    size=8,
    wspace=0.4
)